In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders = spark.table("workspace.default.bronze_orders")

orders = orders.filter(F.col("order_id").isNotNull())

window_spec = Window.partitionBy("order_id").orderBy(
    F.col("order_purchase_timestamp").desc_nulls_last()
)

orders = (
    orders
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

is_delivered = F.col("order_status") == "delivered"

valid_delivery = (
    is_delivered &
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_purchase_timestamp").isNotNull()
)

valid_delay = (
    valid_delivery &
    F.col("order_estimated_delivery_date").isNotNull()
)

orders = (
    orders
    .withColumn(
        "delivery_duration_days",
        F.when(
            valid_delivery,
            F.round(
                (F.unix_timestamp("order_delivered_customer_date") -
                 F.unix_timestamp("order_purchase_timestamp")) / 86400, 2
            )
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            valid_delay,
            F.round(
                (F.unix_timestamp("order_delivered_customer_date") -
                 F.unix_timestamp("order_estimated_delivery_date")) / 86400, 2
            )
        )
    )
    .withColumn(
        "delivery_performance",
        F.when(~is_delivered, "Not Delivered")
        .when(F.col("order_delivered_customer_date").isNull(), "Missing Delivery Date")
        .when(F.col("order_purchase_timestamp").isNull(), "Missing Purchase Date")
        .when(F.col("order_estimated_delivery_date").isNull(), "Missing Estimate")
        .when(F.col("delivery_delay_days") > 0, "Late")
        .when(F.col("delivery_delay_days") < 0, "Early")
        .otherwise("On Time")
    )
    .withColumn(
        "extreme_delay_flag",
        F.when(F.col("delivery_delay_days") > 90, True).otherwise(False)
    )
)

(
    orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_orders")
)

print("Silver orders updated:", orders.count())

Silver orders updated: 99441


Validate our Silver table

In [0]:

display(
    spark.sql("""
        SELECT
            delivery_performance,
            COUNT(*) AS total_orders,
            ROUND(AVG(delivery_delay_days), 2) AS avg_delay_days,
            ROUND(AVG(delivery_duration_days), 2) AS avg_delivery_days
        FROM workspace.default.silver_orders
        GROUP BY delivery_performance
        ORDER BY total_orders DESC
    """)
)


delivery_performance,total_orders,avg_delay_days,avg_delivery_days
Early,88634,-13.01,10.88
Late,7820,9.56,31.53
Not Delivered,2963,null,null
On Time,16,0.0,19.24
Missing Delivery Date,8,null,null


Validate delivery calculations

In [0]:

display(
    spark.sql("""
        SELECT
            order_id,
            order_status,
            order_purchase_timestamp,
            order_delivered_customer_date,
            order_estimated_delivery_date,
            delivery_duration_days,
            delivery_delay_days,
            delivery_performance,
            extreme_delay_flag
        FROM workspace.default.silver_orders
        WHERE order_status = 'delivered'
        ORDER BY delivery_delay_days DESC
        LIMIT 10
    """)
)


order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_duration_days,delivery_delay_days,delivery_performance,extreme_delay_flag
1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23T14:57:35.000Z,2018-09-19T23:24:07.000Z,2018-03-15T00:00:00.000Z,208.35,188.98,Late,true
ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21T23:31:27.000Z,2017-09-19T14:36:39.000Z,2017-03-22T00:00:00.000Z,209.63,181.61,Late,true
47b40429ed8cce3aee9199792275433f,delivered,2018-01-03T09:44:01.000Z,2018-07-13T20:51:31.000Z,2018-01-19T00:00:00.000Z,191.46,175.87,Late,true
2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13T20:17:10.000Z,2017-09-19T17:00:07.000Z,2017-04-05T00:00:00.000Z,189.86,167.71,Late,true
285ab9426d6982034523a855f55a885e,delivered,2017-03-08T22:47:40.000Z,2017-09-19T14:00:04.000Z,2017-04-06T00:00:00.000Z,194.63,166.58,Late,true
440d0d17af552815d15a9e41abe49359,delivered,2017-03-07T23:59:51.000Z,2017-09-19T15:12:50.000Z,2017-04-07T00:00:00.000Z,195.63,165.63,Late,true
c27815f7e3dd0b926b58552628481575,delivered,2017-03-15T23:23:17.000Z,2017-09-19T17:14:25.000Z,2017-04-10T00:00:00.000Z,187.74,162.72,Late,true
d24e8541128cea179a11a65176e0a96f,delivered,2017-06-12T13:14:11.000Z,2017-12-04T18:36:29.000Z,2017-06-26T00:00:00.000Z,175.22,161.78,Late,true
0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09T13:26:57.000Z,2017-09-19T14:38:21.000Z,2017-04-11T00:00:00.000Z,194.05,161.61,Late,true
2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15T11:24:27.000Z,2017-09-19T14:38:18.000Z,2017-04-13T00:00:00.000Z,188.13,159.61,Late,true


Investigate extreme delays

In [0]:

display(
    spark.sql("""
        SELECT
            COUNT(*) AS delivered_orders,
            SUM(CASE WHEN delivery_delay_days > 30 THEN 1 ELSE 0 END) AS delayed_over_30_days,
            SUM(CASE WHEN delivery_delay_days > 60 THEN 1 ELSE 0 END) AS delayed_over_60_days,
            SUM(CASE WHEN delivery_delay_days > 90 THEN 1 ELSE 0 END) AS delayed_over_90_days,
            MAX(delivery_delay_days) AS maximum_delay_days,
            MIN(delivery_duration_days) AS minimum_delivery_days
        FROM workspace.default.silver_orders
        WHERE order_status = 'delivered'
    """)
)


delivered_orders,delayed_over_30_days,delayed_over_60_days,delayed_over_90_days,maximum_delay_days,minimum_delivery_days
96478,360,84,47,188.98,0.53


Check the extreme records

In [0]:
%sql
SELECT
    DATE(order_delivered_customer_date) AS delivery_date,
    COUNT(*) AS orders_delivered,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delay_days,
    MAX(delivery_delay_days) AS maximum_delay_days
FROM workspace.default.silver_orders
WHERE order_status = 'delivered'
  AND delivery_delay_days > 90
GROUP BY DATE(order_delivered_customer_date)
ORDER BY orders_delivered DESC
LIMIT 10;

delivery_date,orders_delivered,avg_delay_days,maximum_delay_days
2017-09-19,21,141.56,181.61
2018-09-19,1,188.98,188.98
2018-04-28,1,94.66,94.66
2017-08-28,1,152.68,152.68
2017-10-26,1,121.87,121.87
2018-04-16,1,110.98,110.98
2018-10-11,1,106.7,106.7
2018-03-26,1,105.55,105.55
2018-04-19,1,104.89,104.89
2018-07-13,1,175.87,175.87


September 19 records

In [0]:
%sql
SELECT
    order_id,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    delivery_duration_days,
    delivery_delay_days
FROM workspace.default.silver_orders
WHERE DATE(order_delivered_customer_date) = '2017-09-19'
ORDER BY delivery_delay_days DESC;

order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_duration_days,delivery_delay_days
ca07593549f1816d26a572e06dc1eab6,2017-02-21T23:31:27.000Z,2017-02-23T02:35:15.000Z,2017-03-08T13:47:46.000Z,2017-09-19T14:36:39.000Z,2017-03-22T00:00:00.000Z,209.63,181.61
2fe324febf907e3ea3f2aa9650869fa5,2017-03-13T20:17:10.000Z,2017-03-13T20:17:10.000Z,2017-03-17T07:23:53.000Z,2017-09-19T17:00:07.000Z,2017-04-05T00:00:00.000Z,189.86,167.71
285ab9426d6982034523a855f55a885e,2017-03-08T22:47:40.000Z,2017-03-08T22:47:40.000Z,2017-03-09T08:53:20.000Z,2017-09-19T14:00:04.000Z,2017-04-06T00:00:00.000Z,194.63,166.58
440d0d17af552815d15a9e41abe49359,2017-03-07T23:59:51.000Z,2017-03-09T01:11:33.000Z,2017-03-15T13:00:08.000Z,2017-09-19T15:12:50.000Z,2017-04-07T00:00:00.000Z,195.63,165.63
c27815f7e3dd0b926b58552628481575,2017-03-15T23:23:17.000Z,2017-03-15T23:23:17.000Z,2017-03-21T14:50:25.000Z,2017-09-19T17:14:25.000Z,2017-04-10T00:00:00.000Z,187.74,162.72
0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09T13:26:57.000Z,2017-03-09T13:26:57.000Z,2017-03-22T05:28:14.000Z,2017-09-19T14:38:21.000Z,2017-04-11T00:00:00.000Z,194.05,161.61
2d7561026d542c8dbd8f0daeadf67a43,2017-03-15T11:24:27.000Z,2017-03-15T11:24:27.000Z,2017-03-16T09:29:16.000Z,2017-09-19T14:38:18.000Z,2017-04-13T00:00:00.000Z,188.13,159.61
2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08T18:09:02.000Z,2017-03-08T18:09:02.000Z,2017-03-13T08:54:02.000Z,2017-09-19T14:33:17.000Z,2017-04-17T00:00:00.000Z,194.85,155.61
dfe5f68118c2576143240b8d78e5940a,2017-03-17T12:32:22.000Z,2017-03-17T12:32:22.000Z,2017-03-21T18:28:04.000Z,2017-09-19T18:13:19.000Z,2017-04-19T00:00:00.000Z,186.24,153.76
437222e3fd1b07396f1d9ba8c15fba59,2017-03-16T11:36:00.000Z,2017-03-16T11:36:00.000Z,2017-03-21T08:22:43.000Z,2017-09-19T16:28:58.000Z,2017-04-28T00:00:00.000Z,187.2,144.69


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_purchase_timestamp) AS unique_purchase_timestamps,
    COUNT(DISTINCT order_delivered_customer_date) AS unique_delivery_timestamps,
    MIN(order_purchase_timestamp) AS earliest_purchase,
    MAX(order_purchase_timestamp) AS latest_purchase
FROM workspace.default.silver_orders
WHERE DATE(order_delivered_customer_date) = '2017-09-19';

total_orders,unique_purchase_timestamps,unique_delivery_timestamps,earliest_purchase,latest_purchase
282,282,280,2017-02-21T23:31:27.000Z,2017-09-18T15:07:49.000Z


Validation

In [0]:
%sql
SELECT
    delivery_performance,
    COUNT(*) AS total_orders,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delay_days,
    ROUND(AVG(delivery_duration_days), 2) AS avg_delivery_days
FROM workspace.default.silver_orders
GROUP BY delivery_performance
ORDER BY total_orders DESC;

delivery_performance,total_orders,avg_delay_days,avg_delivery_days
Early,88634,-13.01,10.88
Late,7820,9.56,31.53
Not Delivered,2963,null,null
On Time,16,0.0,19.24
Missing Delivery Date,8,null,null


Verify our extreme-delay flags


In [0]:
%sql
SELECT
    extreme_delay_flag,
    COUNT(*) AS total_orders
FROM workspace.default.silver_orders
GROUP BY extreme_delay_flag;

extreme_delay_flag,total_orders
false,99394
true,47


Build Silver Customers

In [0]:
from pyspark.sql import functions as F

customers = spark.table("workspace.default.bronze_customers")

customers = customers.filter(
    F.col("customer_id").isNotNull()
)

customers = customers.dropDuplicates(["customer_id"])

customers = customers.withColumn(
    "customer_zip_prefix",
    F.lpad(F.col("customer_zip_code_prefix").cast("string"), 5, "0")
)

(
    customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_customers")
)

print("Silver customers created:", customers.count())

Silver customers created: 99441


Customer validation

In [0]:
%sql
SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(customer_zip_prefix) AS valid_zip_codes,
    COUNT(*) - COUNT(customer_zip_prefix) AS missing_zip_codes
FROM workspace.default.silver_customers;

total_customers,unique_customers,valid_zip_codes,missing_zip_codes
99441,99441,99441,0


Silver Sellers

In [0]:
from pyspark.sql import functions as F

sellers = spark.table("workspace.default.bronze_sellers")

sellers = sellers.filter(
    F.col("seller_id").isNotNull()
)

sellers = sellers.dropDuplicates(["seller_id"])

sellers = sellers.withColumn(
    "seller_zip_prefix",
    F.lpad(F.col("seller_zip_code_prefix").cast("string"), 5, "0")
)

(
    sellers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_sellers")
)

print("Silver sellers created:", sellers.count())

Silver sellers created: 3095


Seller validation

In [0]:
%sql
SELECT
    COUNT(*) AS total_sellers,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    COUNT(seller_zip_prefix) AS valid_zip_codes,
    COUNT(*) - COUNT(seller_zip_prefix) AS missing_zip_codes
FROM workspace.default.silver_sellers;

total_sellers,unique_sellers,valid_zip_codes,missing_zip_codes
3095,3095,3095,0


Silver Order Items

In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.default.bronze_order_items")

items = items.filter(
    F.col("order_id").isNotNull() &
    F.col("order_item_id").isNotNull() &
    F.col("seller_id").isNotNull()
)

items = items.dropDuplicates(["order_id", "order_item_id"])

items = (
    items
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .withColumn(
        "total_item_cost",
        F.round(F.col("price") + F.col("freight_value"), 2)
    )
)

(
    items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_order_items")
)

print("Silver order items created:", items.count())

Silver order items created: 112650


Validate item data

In [0]:
%sql
SELECT
    COUNT(*) AS total_items,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS negative_prices,
    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS negative_freight
FROM workspace.default.silver_order_items;

total_items,unique_orders,unique_sellers,negative_prices,negative_freight
112650,98666,3095,0,0


Silver Payments

In [0]:
from pyspark.sql import functions as F

payments = spark.table("workspace.default.bronze_payments")

payments = payments.filter(
    F.col("order_id").isNotNull()
)

payments = (
    payments
    .withColumn("payment_value", F.col("payment_value").cast("double"))
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )
)

(
    payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_payments")
)

print("Silver payments created:", payments.count())

Silver payments created: 103886


Payment validation

In [0]:
%sql
SELECT
    payment_type,
    COUNT(*) AS total_payments,
    ROUND(SUM(payment_value), 2) AS total_value,
    ROUND(AVG(payment_value), 2) AS avg_payment
FROM workspace.default.silver_payments
GROUP BY payment_type
ORDER BY total_payments DESC;

payment_type,total_payments,total_value,avg_payment
credit_card,76795,1.254208419E7,163.32
boleto,19784,2869361.27,145.03
voucher,5775,379436.87,65.7
debit_card,1529,217989.79,142.57
not_defined,3,0.0,0.0


Silver Products

In [0]:
from pyspark.sql import functions as F

products = spark.table("workspace.default.bronze_products")

products = products.filter(
    F.col("product_id").isNotNull()
)

products = products.dropDuplicates(["product_id"])

products = (
    products
    .withColumn(
        "product_category_name",
        F.coalesce(
            F.col("product_category_name"),
            F.lit("unknown")
        )
    )
    .withColumn(
        "product_weight_g",
        F.col("product_weight_g").cast("double")
    )
    .withColumn(
        "product_volume_cm3",
        F.col("product_length_cm").cast("double") *
        F.col("product_height_cm").cast("double") *
        F.col("product_width_cm").cast("double")
    )
)

(
    products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_products")
)

print("Silver products created:", products.count())

Silver products created: 32951


Product validation

In [0]:
%sql
SELECT
    COUNT(*) AS total_products,
    SUM(CASE WHEN product_category_name = 'unknown' THEN 1 ELSE 0 END) AS unknown_categories,
    SUM(CASE WHEN product_weight_g IS NULL THEN 1 ELSE 0 END) AS missing_weight,
    SUM(CASE WHEN product_volume_cm3 IS NULL THEN 1 ELSE 0 END) AS missing_volume,
    SUM(CASE WHEN product_weight_g <= 0 THEN 1 ELSE 0 END) AS invalid_weight,
    SUM(CASE WHEN product_volume_cm3 <= 0 THEN 1 ELSE 0 END) AS invalid_volume
FROM workspace.default.silver_products;

total_products,unknown_categories,missing_weight,missing_volume,invalid_weight,invalid_volume
32951,610,2,2,4,0


In [0]:
from pyspark.sql import functions as F

categories = spark.table("workspace.default.bronze_category_translation")

categories = (
    categories
    .filter(F.col("product_category_name").isNotNull())
    .dropDuplicates(["product_category_name"])
    .withColumnRenamed(
        "product_category_name_english",
        "category_english"
    )
)

(
    categories.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_category_translation")
)

print("Silver categories created:", categories.count())

Silver categories created: 71


Verify translations

In [0]:
%sql
SELECT
    COUNT(*) AS total_categories,
    COUNT(category_english) AS translated_categories,
    COUNT(*) - COUNT(category_english) AS missing_translations
FROM workspace.default.silver_category_translation;

total_categories,translated_categories,missing_translations
71,71,0


In [0]:
%sql
SELECT *
FROM workspace.default.bronze_geolocation
LIMIT 10;

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP
1012,-23.547762303364266,-46.63536053788448,são paulo,SP
1047,-23.546273112412678,-46.64122516971552,sao paulo,SP
1013,-23.546923208436723,-46.6342636964915,sao paulo,SP
1029,-23.543769055769133,-46.63427784085132,sao paulo,SP
1011,-23.547639550320632,-46.63603162315495,sao paulo,SP


Clean Geolocation

In [0]:
from pyspark.sql import functions as F

geo = spark.table("workspace.default.bronze_geolocation")

geo = geo.filter(
    F.col("geolocation_zip_code_prefix").isNotNull() &
    F.col("geolocation_lat").isNotNull() &
    F.col("geolocation_lng").isNotNull()
)

geo = geo.withColumn(
    "zip_prefix",
    F.lpad(
        F.col("geolocation_zip_code_prefix").cast("string"),
        5,
        "0"
    )
)

geo = (
    geo.groupBy("zip_prefix")
    .agg(
        F.avg("geolocation_lat").alias("latitude"),
        F.avg("geolocation_lng").alias("longitude"),
        F.first("geolocation_city").alias("city"),
        F.first("geolocation_state").alias("state"),
        F.count("*").alias("location_records")
    )
)

(
    geo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_geolocation")
)

print("Silver geolocation created:", geo.count())

Silver geolocation created: 19015


Validate Geolocation

In [0]:
%sql
SELECT
    COUNT(*) AS total_locations,
    COUNT(DISTINCT zip_prefix) AS unique_zip_codes,
    SUM(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 ELSE 0 END) AS missing_coordinates,
    SUM(CASE WHEN latitude NOT BETWEEN -90 AND 90
          OR longitude NOT BETWEEN -180 AND 180 THEN 1 ELSE 0 END) AS invalid_coordinates
FROM workspace.default.silver_geolocation;

total_locations,unique_zip_codes,missing_coordinates,invalid_coordinates
19015,19015,0,0


In [0]:
reviews = spark.table("workspace.default.bronze_reviews")

print("Columns:", reviews.columns)
print("Score distribution:")
reviews.groupBy("review_score").count().orderBy("review_score").show(10)

Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
Score distribution:
+--------------------+-----+
|        review_score|count|
+--------------------+-----+
|                NULL| 2380|
|                   "|    1|
| BARALHO KEM COM ...|    1|
|       Desta vez nã"|    1|
|                   M|    1|
| NAO confere para...|    1|
| VOU RECEBER DEPO...|    1|
| a URA não resolv...|    1|
| a previsão dada ...|    1|
| a pulseira do me...|    1|
+--------------------+-----+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

reviews = spark.table("workspace.default.bronze_reviews")

valid = F.expr("try_cast(review_score AS INT)")

print("Total records:", reviews.count())
print("Valid scores:", reviews.filter(valid.between(1, 5)).count())
print("Invalid/missing scores:", reviews.filter(
    valid.isNull() | ~valid.between(1, 5)
).count())

Total records: 104162
Valid scores: 99224
Invalid/missing scores: 4938


In [0]:
reviews.filter(
    F.expr("try_cast(review_score AS INT) BETWEEN 1 AND 5")
).select(
    F.count("*").alias("valid_reviews"),
    F.countDistinct("review_id").alias("unique_review_ids"),
    F.countDistinct("order_id").alias("unique_orders")
).show()

+-------------+-----------------+-------------+
|valid_reviews|unique_review_ids|unique_orders|
+-------------+-----------------+-------------+
|        99224|            98410|        98673|
+-------------+-----------------+-------------+



silver_reviews

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

reviews = spark.table("workspace.default.bronze_reviews")

reviews = (
    reviews
    .withColumn("review_score_clean", F.expr("try_cast(review_score AS INT)"))
    .filter(
        F.col("order_id").isNotNull() &
        F.col("review_score_clean").between(1, 5)
    )
    .withColumn("review_id", F.trim("review_id"))
    .withColumn("review_comment_title", F.trim("review_comment_title"))
    .withColumn("review_comment_message", F.trim("review_comment_message"))
    .withColumn(
        "sentiment",
        F.when(F.col("review_score_clean") <= 2, "Negative")
        .when(F.col("review_score_clean") == 3, "Neutral")
        .otherwise("Positive")
    )
    .drop("review_score")
    .withColumnRenamed("review_score_clean", "review_score")
)

(
    reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_reviews")
)

print("Silver reviews created:", reviews.count())

Silver reviews created: 99224


In [0]:
reviews = spark.table("workspace.default.silver_reviews")

print("Total:", reviews.count())
print("Invalid scores:", reviews.filter(~F.col("review_score").between(1, 5)).count())
print("Missing order IDs:", reviews.filter(F.col("order_id").isNull()).count())

reviews.groupBy("sentiment").count().show()

Total: 99224
Invalid scores: 0
Missing order IDs: 0
+---------+-----+
|sentiment|count|
+---------+-----+
| Positive|76470|
| Negative|14575|
|  Neutral| 8179|
+---------+-----+



Silver Routing table

In [0]:
from pyspark.sql import functions as F

routes = spark.table("workspace.default.bronze_routes")

silver_routes = (
    routes
    .filter(
        (F.col("osrm_code") == "Ok") &
        (F.col("distance_meters") > 0) &
        (F.col("duration_seconds") > 0) &
        F.col("seller_zip").isNotNull() &
        F.col("customer_zip").isNotNull()
    )
    .dropDuplicates(["seller_zip", "customer_zip"])
    .withColumn("seller_zip", F.col("seller_zip").cast("string"))
    .withColumn("customer_zip", F.col("customer_zip").cast("string"))
    .withColumn("distance_km", F.round(F.col("distance_meters") / 1000, 2))
    .withColumn("estimated_duration_hours", F.round(F.col("duration_seconds") / 3600, 2))
    .select(
        "seller_zip",
        "customer_zip",
        "shipment_count",
        "distance_km",
        "estimated_duration_hours",
        "osrm_code"
    )
)

silver_routes.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_routes"
)

print("Silver Routing table created!")
print("Valid routes:", silver_routes.count())
print("Duplicate routes:", silver_routes.count() - silver_routes.dropDuplicates(["seller_zip", "customer_zip"]).count())

display(silver_routes.orderBy(F.desc("shipment_count")))

Silver Routing table created!
Valid routes: 50
Duplicate routes: 0


seller_zip,customer_zip,shipment_count,distance_km,estimated_duration_hours,osrm_code
14940,28970,15,851.75,11.18,Ok
81130,80030,15,14.5,0.35,Ok
14940,35162,14,867.79,12.11,Ok
14940,22793,13,752.73,9.72,Ok
14940,13212,12,288.73,3.8,Ok
14940,11065,10,418.06,5.55,Ok
14940,22775,10,740.26,9.59,Ok
14940,29101,9,1177.47,17.29,Ok
14940,13290,9,275.81,3.7,Ok
14940,33200,9,676.71,8.9,Ok
